In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [3]:
files = sorted(glob.glob('/home/ulyanov/data/stereo/*'))
files

['/home/ulyanov/data/stereo/2024-03-30',
 '/home/ulyanov/data/stereo/2025-09-24',
 '/home/ulyanov/data/stereo/2025-10-07',
 '/home/ulyanov/data/stereo/vlos']

In [4]:
file_hmi = files[1]
file_fdt = files[5]

file_hmi, file_fdt

IndexError: list index out of range

In [9]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

In [13]:
for i, dcrota in enumerate(np.arange(0.25,0.35,0.01)):

    with fits.open(file_hmi) as hdul:
        header_hmi = hdul[1].header
        data_hmi = hdul[1].data

    with fits.open(file_fdt) as hdul:
        header_fdt = hdul[0].header
        data_fdt = hdul[0].data

    did = int(file_fdt.split('_')[-1].split('.')[0])
    data_fdt = undistort(data_fdt, header_fdt, xd, yd)

    data_hmi = rebin(data_hmi, 4, update_header=header_hmi)
    data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=False, mu_thr=0.2,
                         xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                         crota=header_fdt['CROTA'] + dcrota)

    data_fdt[np.isnan(data_hmi)] = np.nan

    q = np.nanmean(data_fdt * data_hmi) / np.sqrt(np.nanmean(data_fdt ** 2) * np.nanmean(data_hmi ** 2))
    #cor[i] = q

    print(dcrota,  q)

0.25 0.9637384550638861
0.26 0.9643503032331705
0.27 0.9647353204459999
0.28 0.9648880010551121
0.29000000000000004 0.9648096554723914
0.30000000000000004 0.9645141866737538
0.31000000000000005 0.963986498783697
0.32000000000000006 0.9632177282038087
0.33000000000000007 0.962239776973921
0.3400000000000001 0.9610361217345222
